# Train a two-neural network model to handle asymptotic behaviour

In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
from DerivPlots import scatter_plot
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt
import QuantLib as ql
from BasketDataGen import value_basket

## Load and Prepare the dataset A

In [ ]:
filename_root = 'test_basket_mt'

In [ ]:
df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

In [ ]:
dfA = df1[df1['samples'] == 100000]

In [ ]:
dfA = dfA.drop('samples', axis=1)

In [ ]:
yA = dfA['option_value']

In [ ]:
XA = dfA.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

In [ ]:
XA_train, XA_test, yA_train, yA_test = train_test_split(XA, yA, test_size=0.01) 

## Load and Prepare the dataset C (dispersed)

In [ ]:
filename_root = 'test_basket_disp'

In [ ]:
import pandas as pd

dfs = []

for i in range(20):
    filename = f"{filename_root}{i}.csv"
    df = pd.read_csv(filename)
    dfs.append(df)

df1 = pd.concat(dfs, ignore_index=False)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

# Drop unwanted column (only if it exists)
df1 = df1.drop(columns=['Unnamed: 0'], errors='ignore')

# Filter rows
dfC = df1[df1['samples'] == 1000].copy()

# Drop columns
dfC = dfC.drop(columns=['samples'])

# Target and features
yC = dfC['option_value']
XC = dfC.drop(columns=['option_value', 'process_time', 'error_estimate'])

In [ ]:
XC_train, XC_test, yC_train, yC_test = train_test_split(XC, yC, test_size=0.01) 

In [ ]:
scaler = StandardScaler()
XC_train = scaler.fit_transform(XC_train)
XC_test = scaler.transform(XC_test)
XA_train = scaler.transform(XA_train)
XA_test = scaler.transform(XA_test)

## Train model coarse model

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
torch_tensor = torch.from_numpy(XC_train).float().to(device)
y_tensor = torch.from_numpy(yC_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(XC_test).float().to(device)
test_y_tensor = torch.from_numpy(yC_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
epochs = 200
lr = 0.001

In [ ]:
modelC = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizer = optim.Adam(modelC.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
modelC_out = train_model(modelC, data_loader, test_data_loader, loss_fn, optimizer, epochs)

#### Stock Price test

In [ ]:
column_names = XC.columns.tolist()

maturity = 365
stock_price0 = 150.0
vols0 = 0.5
stock_price1 = 150.0
vols1 = 0.5
stock_price2 = 150.0
vols2 = 0.5
stock_price3 = 150.0
vols3 = 0.5
stock_price4 = 150.0
vols4 = 0.5
stock_price5 = 150.0
vols5 = 0.5
rho_0_1 = 0.4
rho_0_2 = 0.4
rho_0_3 = 0.4
rho_0_4 = 0.4
rho_0_5 = 0.4
rho_1_2 = 0.4
rho_1_3 = 0.4
rho_1_4 = 0.4
rho_1_5 = 0.4
rho_2_3 = 0.4
rho_2_4 = 0.4
rho_2_5 = 0.4
rho_3_4 = 0.4
rho_3_5 = 0.4
rho_4_5 = 0.4

input_dict = dict()

# Iterate through the variable names and add them to the dictionary
for variable_name in column_names:
    # Use globals() to access the value associated with the variable name
    input_dict[variable_name] = globals().get(variable_name)

In [ ]:
stock_price_override = np.linspace(0.0, 400.0, 17, endpoint=True)
input_df = pd.DataFrame([input_dict] * 17)
input_df['stock_price0'] = stock_price_override
input_scaled = scaler.transform(input_df)
NN_res = modelC(torch.tensor(input_scaled, dtype=torch.float32).to(device))
NN_res_np = NN_res.detach().cpu().numpy()

In [ ]:
interest_rate = 0.0
strike = 100.0
divs = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
mc_samples_size = [100000]

In [ ]:
MC_res = list()

for index, row in input_df.iterrows():
    maturity = int(row['maturity'])
    stocks = [row['stock_price0'], row['stock_price1'], row['stock_price2'], 
              row['stock_price3'], row['stock_price4'], row['stock_price5']]
    vols = [row['vols0'], row['vols1'], row['vols2'], 
            row['vols3'], row['vols4'], row['vols5']]
    corr_matrix = ql.Matrix(6,6,0.4)
    corr_matrix[0][0] = 1.0
    corr_matrix[1][1] = 1.0
    corr_matrix[2][2] = 1.0
    corr_matrix[3][3] = 1.0
    corr_matrix[4][4] = 1.0
    corr_matrix[5][5] = 1.0
    npv = value_basket(interest_rate, maturity, strike, stocks, vols, divs, corr_matrix, mc_samples_size) 
    MC_res.append(npv[0]['option_value'])

In [ ]:
plt_filename = 'stock_pricednnmc_asym.png'

plt.figure(figsize=(10, 6))
plt.plot(stock_price_override, NN_res_np, color='black', linestyle='-', label='DNN values')
plt.plot(stock_price_override, MC_res, color='black', linestyle='--', label='MC values')
plt.xlabel('Stock Price')
plt.ylabel('Option NPV')
plt.legend()
plt.tight_layout()
plt.savefig(plt_filename, dpi=300)
plt.show()

In [ ]:
filename_root_acc = 'test_basket_accuracy'

dfs_acc = []

for i in range(20):
    filenameacc = f"{filename_root_acc}{i}.csv"
    df = pd.read_csv(filenameacc)
    dfs_acc.append(df)

df1acc = pd.concat(dfs_acc, ignore_index=False)

df1acc = df1acc.drop(columns=['Unnamed: 0'], errors='ignore')

df1acc1e7 = df1acc[df1acc['samples'] == 10_000_000].copy()

df1acc1e7 = df1acc1e7.drop(columns=['samples'])

yacc = df1acc1e7['option_value']
Xacc = df1acc1e7.drop(columns=['option_value', 'process_time', 'error_estimate'])

X_acc = scaler.transform(Xacc)

test_torch_tensoracc = torch.from_numpy(X_acc).float().to(device)
test_y_tensoracc = torch.from_numpy(yacc.to_numpy()).float().to(device)

test_datasetacc = TensorDataset(test_torch_tensoracc, test_y_tensoracc)
test_data_loaderacc = DataLoader(
    test_datasetacc,
    batch_size=1000,
    shuffle=False,
    drop_last=True
)

In [ ]:
modelC.eval()
test_mse_listC = list()
C_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelC(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        C_errors = np.append(C_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listC.append(mse)

In [ ]:
test_avg_mse_C = sum(test_mse_listC) / len(test_mse_listC)
print(test_avg_mse_C)

In [ ]:
torch.save(modelC, 'modelCMCasym.pt')

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelCasym_scatter')

In [ ]:
x_max = 1.5
x_min = -1.5
bin_edges = np.linspace(x_min, x_max, num=40)

In [ ]:
# Create a histogram plot
plt.hist(C_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('modelCasymrrors.png', dpi=300)
plt.show()

## Fine tune

In [ ]:
torch_tensor = torch.from_numpy(XA_train).float().to(device)
y_tensor = torch.from_numpy(yA_train.to_numpy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(XA_test).float().to(device)
test_y_tensor = torch.from_numpy(yA_test.to_numpy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
tune_epochs = 50

In [ ]:
modelC_out = train_model(modelC, data_loader, test_data_loader, loss_fn, optimizer, tune_epochs)

In [ ]:
column_names = XC.columns.tolist()

maturity = 365
stock_price0 = 150.0
vols0 = 0.5
stock_price1 = 150.0
vols1 = 0.5
stock_price2 = 150.0
vols2 = 0.5
stock_price3 = 150.0
vols3 = 0.5
stock_price4 = 150.0
vols4 = 0.5
stock_price5 = 150.0
vols5 = 0.5
rho_0_1 = 0.4
rho_0_2 = 0.4
rho_0_3 = 0.4
rho_0_4 = 0.4
rho_0_5 = 0.4
rho_1_2 = 0.4
rho_1_3 = 0.4
rho_1_4 = 0.4
rho_1_5 = 0.4
rho_2_3 = 0.4
rho_2_4 = 0.4
rho_2_5 = 0.4
rho_3_4 = 0.4
rho_3_5 = 0.4
rho_4_5 = 0.4

input_dict = dict()

# Iterate through the variable names and add them to the dictionary
for variable_name in column_names:
    # Use globals() to access the value associated with the variable name
    input_dict[variable_name] = globals().get(variable_name)

stock_price_override = np.linspace(0.0, 400.0, 17, endpoint=True)
input_df = pd.DataFrame([input_dict] * 17)
input_df['stock_price0'] = stock_price_override
input_scaled = scaler.transform(input_df)
NN_res = modelC(torch.tensor(input_scaled, dtype=torch.float32).to(device))
NN_res_np = NN_res.detach().cpu().numpy()

In [ ]:
interest_rate = 0.0
strike = 100.0
divs = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
mc_samples_size = [100000]

MC_res = list()

for index, row in input_df.iterrows():
    maturity = int(row['maturity'])
    stocks = [row['stock_price0'], row['stock_price1'], row['stock_price2'], 
              row['stock_price3'], row['stock_price4'], row['stock_price5']]
    vols = [row['vols0'], row['vols1'], row['vols2'], 
            row['vols3'], row['vols4'], row['vols5']]
    corr_matrix = ql.Matrix(6,6,0.4)
    corr_matrix[0][0] = 1.0
    corr_matrix[1][1] = 1.0
    corr_matrix[2][2] = 1.0
    corr_matrix[3][3] = 1.0
    corr_matrix[4][4] = 1.0
    corr_matrix[5][5] = 1.0
    npv = value_basket(interest_rate, maturity, strike, stocks, vols, divs, corr_matrix, mc_samples_size) 
    MC_res.append(npv[0]['option_value'])

In [ ]:
plt_filename = 'stock_pricednnmc_asymtuned.png'

plt.figure(figsize=(10, 6))
plt.plot(stock_price_override, NN_res_np, color='black', linestyle='-', label='DNN values')
plt.plot(stock_price_override, MC_res, color='black', linestyle='--', label='MC values')
plt.xlabel('Stock Price')
plt.ylabel('Option NPV')
plt.legend()
plt.tight_layout()
plt.savefig(plt_filename, dpi=300)
plt.show()

In [ ]:
filename_root_acc = 'test_basket_accuracy'

dfs_acc = []
for i in range(20):
    filenameacc = f"{filename_root_acc}{i}.csv"
    dfs_acc.append(pd.read_csv(filenameacc))

df1acc = pd.concat(dfs_acc, ignore_index=False)

df1acc = df1acc.drop(columns=['Unnamed: 0'], errors='ignore')

df1acc1e7 = df1acc[df1acc['samples'] == 10_000_000].copy()
df1acc1e7 = df1acc1e7.drop(columns=['samples'])

yacc = df1acc1e7['option_value']
Xacc = df1acc1e7.drop(columns=['option_value', 'process_time', 'error_estimate'])

X_acc = scaler.transform(Xacc)

test_torch_tensoracc = torch.from_numpy(X_acc).float().to(device)
test_y_tensoracc = torch.from_numpy(yacc.to_numpy()).float().to(device)

test_datasetacc = TensorDataset(test_torch_tensoracc, test_y_tensoracc)
test_data_loaderacc = DataLoader(
    test_datasetacc,
    batch_size=1000,
    shuffle=False,
    drop_last=False
)

modelC.eval()
test_mse_listC = []
C_errors = []

with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelC(batch_X).squeeze()
        mse = F.mse_loss(test_outputs, batch_y).item()
        errors = test_outputs.cpu().numpy() - batch_y.cpu().numpy()

        test_mse_listC.append(mse)
        C_errors.append(errors)

C_errors = np.concatenate(C_errors)

In [ ]:
test_avg_mse_C = sum(test_mse_listC) / len(test_mse_listC)
print(test_avg_mse_C)

In [ ]:
torch.save(modelC, 'modelCMCasymtuned.pt')

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelCasymtuned_scatter')

In [ ]:
# Create a histogram plot
plt.hist(C_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('modelCasymtunedrrors.png', dpi=300)
plt.show()